# Paper 18 · U-Net

**Citation:** Olaf Ronneberger, Philipp Fischer, Thomas Brox, “U-Net: Convolutional Networks for Biomedical Image Segmentation” (2015).

**Paper:** https://arxiv.org/abs/1505.04597

> **Scale gap:** We segment synthetic circles in 32×32 images with a tiny U-Net, not biomedical datasets.

## Mathematical Framework

Before reproducing the paper experimentally, work through:

- [Math 01 · Linear Algebra & Geometry](../../math/01_linear_algebra_geometry.ipynb)
- [Math 02 · Calculus & Matrix Calculus](../../math/02_calculus_matrix_calculus.ipynb)
- [Math 10 · Neural-Network Mathematics](../../math/10_neural_network_math.ipynb)

Your explanation should connect the paper's empirical claim to its **mathematical objective, representation, assumptions, and optimization/statistical argument**.

## Before you read
1. Why does downsampling lose localization detail?
2. What information do skip connections restore?
3. Why is segmentation different from image-level classification?

## Central claim
An encoder-decoder with skip connections can combine coarse contextual features with fine spatial detail for dense segmentation.

## Synthetic segmentation data

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


In [ ]:
from coursekit.experiments import Experiment
experiment = Experiment('paper-18_unet', {'bootstrap_seed': SEED, 'scope': 'educational mechanism demonstration', 'note': 'Original notebook may use additional explicit seeds; source hash records the exact experiment.'}, source='papers/notebooks/18_unet.ipynb')
experiment.capture_figures()

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from torch import nn
torch.manual_seed(0)
def make_batch(n,size=32):
    yy,xx=torch.meshgrid(torch.arange(size),torch.arange(size),indexing="ij")
    imgs=[]; masks=[]
    for _ in range(n):
        cx=int(torch.randint(8,size-8,(1,))); cy=int(torch.randint(8,size-8,(1,))); r=int(torch.randint(3,8,(1,)))
        mask=((xx-cx)**2+(yy-cy)**2<=r*r).float()
        img=(mask+.25*torch.randn(size,size)).clamp(0,1)
        imgs.append(img); masks.append(mask)
    return torch.stack(imgs)[:,None],torch.stack(masks)[:,None]
X,Y=make_batch(300); Xtr,Ytr=X[:240],Y[:240]; Xte,Yte=X[240:],Y[240:]

## Tiny U-Net

In [ ]:
class TinyUNet(nn.Module):
    def __init__(self,use_skip=True):
        super().__init__(); self.use_skip=use_skip
        self.e1=nn.Sequential(nn.Conv2d(1,8,3,padding=1),nn.ReLU(),nn.Conv2d(8,8,3,padding=1),nn.ReLU())
        self.pool=nn.MaxPool2d(2)
        self.e2=nn.Sequential(nn.Conv2d(8,16,3,padding=1),nn.ReLU())
        self.up=nn.ConvTranspose2d(16,8,2,stride=2)
        self.dec=nn.Sequential(nn.Conv2d(16 if use_skip else 8,8,3,padding=1),nn.ReLU(),nn.Conv2d(8,1,1))
    def forward(self,x):
        s=self.e1(x); h=self.e2(self.pool(s)); u=self.up(h)
        if self.use_skip: u=torch.cat([u,s],1)  # TODO: explain this shape.
        return self.dec(u)

def train(use_skip):
    torch.manual_seed(0); m=TinyUNet(use_skip); opt=torch.optim.Adam(m.parameters(),lr=.01); bce=nn.BCEWithLogitsLoss()
    for _ in range(120):
        logits=m(Xtr); loss=bce(logits,Ytr); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        p=(torch.sigmoid(m(Xte))>.5).float()
        inter=(p*Yte).sum((1,2,3)); union=((p+Yte)>0).float().sum((1,2,3))
        iou=(inter/(union+1e-8)).mean().item()
    return m,iou
ms,ious=train(True); mn,ioun=train(False)
print("IoU with skip",ious,"without skip",ioun)

## Visualize one result

In [ ]:
with torch.no_grad(): pred=torch.sigmoid(ms(Xte[:1]))[0,0]
fig,axs=plt.subplots(1,3,figsize=(8,3))
for ax,img,title in zip(axs,[Xte[0,0],Yte[0,0],pred],["input","target","prediction"]):
    ax.imshow(img,cmap="gray"); ax.set_title(title); ax.axis("off")
plt.show()

### Ablation
Remove the skip connection or replace concatenation with addition. Compare IoU and boundary quality.

## Ablation table

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
1. What problem existed before this work?
2. What was actually new?
3. What evidence did this notebook reproduce?
4. What does the scale gap prevent you from claiming?
5. Which contribution remains important today?
6. What would you test next?

## Evidence export

Figures and numeric diagnostics are captured. Explicit metrics use `experiment.log(variant, seed, metrics)`. Use `run_trials` for paired-seed ablations. An empty metrics table or `not_run` ablation is incomplete evidence, not success. Interpretations remain your work.

In [ ]:
print('Evidence directory:', experiment.finish(globals()))